# 01 — Parse, exclusions and splits (M1, redesigned for exploration X2)

**Goal:** turn the raw tarball into one parsed table, drop the excluded molecules, and build every split that
later steps train, develop and test on.

The split design changed in exploration X2 (DECISIONS.md, 2026-10-06). X1's design served only the quantum model
(a 25-per-formula pool of 8,438 molecules, a 7,131-molecule training pool, test sets of 370 and 80 molecules).
Track A, the accurate classical model, needs the data QM9 actually has:

| Set | What it is | Used by |
|---|---|---|
| `test_unseen` | every molecule of a stratified 15% of formulas | `scripts/final_eval.py` only (M7) |
| `test_familiar` | 5% of the remaining molecules; every formula keeps molecules in training | `scripts/final_eval.py` only |
| `dev_unseen` | every molecule of 5% of the formulas left after the test holdout | development: new-formula error |
| `dev` | 5,000 random molecules of the remaining formulas | development: learning curves, diagnostics |
| `pool` (P) | everything else | training; nested sets S_(s,N), N = 100 … full pool |
| `test_unseen_q`, `test_familiar_q` | small fixed subsets of the two test sets | the quantum model (inference costs n_test × N circuits) |

M1 acceptance criteria (PLAN §9.4) still hold: parsed table saved and parser tested (`tests/test_parser.py`);
exclusions applied and documented (`docs/DATA.md`); every split file written to `splits/`, every split check
passes, and a split report is printed. Settings live in `configs/splits.yaml`. Every cell is safe to re-run.

In [ ]:
# Environment check: stops here, with setup instructions, unless this kernel is the project
# environment (README.md, "Setup"). Version differences from requirements.txt only warn.
try:
    from qm9dipole.provenance import check_environment
except ModuleNotFoundError as e:
    raise RuntimeError(
        "The qm9dipole package is not installed in this kernel's environment. Follow "
        "README.md (Setup), then select that environment as the notebook kernel."
    ) from e

print(check_environment())

## 1. Parse the tarball into one table

The tarball is streamed (not extracted) and parsed once into `data/processed/qm9.parquet`. Re-runs load the parquet
in about a second. The table is rebuilt automatically if it was written by an older version of the parser.

| Role | Columns |
|---|---|
| **Headline inputs** (the brief: composition + geometry) | `Z` (atomic numbers), `R` (coordinates, Å) |
| **Target** | `mu`: dipole magnitude in debye |
| **Exploration only**, reported separately (CLAUDE.md, "Exploration") | `q` (Mulliken charges, e), 14 more DFT properties (`alpha`, `homo`, `lumo`, …), `freqs` |
| Bookkeeping | `id`, `formula`, `n_atoms`, `n_heavy`, `smiles`, `inchi`, … |

The exploration columns come from the same DFT calculation as μ, so a model that uses them answers a different
question ("given the electron density, what is μ?") from the brief's ("given the structure, what is μ?").

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from qm9dipole import REPO_ROOT
from qm9dipole.data import (
    HEADLINE_INPUTS, QM9_PARQUET, TARGET, build_qm9_table, load_qm9_table, qm9_table_is_current,
    save_qm9_table,
)
from qm9dipole.plots import INK_SECONDARY, SERIES, SURFACE, use_style

use_style()
REBUILD = False  # set True to force a rebuild

if REBUILD or not qm9_table_is_current():
    table, failures = build_qm9_table()
    print(f"parsed {len(table):,} molecules; {len(failures)} parse failures")
    for name, err in failures:
        print("  ", name, err)
    assert not failures, "parse failures must be fixed or documented in docs/DATA.md"
    save_qm9_table(table)

table = load_qm9_table()
print(f"{len(table):,} molecules loaded from {QM9_PARQUET.relative_to(REPO_ROOT).as_posix()} "
      f"({QM9_PARQUET.stat().st_size / 1e6:.1f} MB, {table.shape[1]} columns)")
print(f"headline inputs: {', '.join(HEADLINE_INPUTS)}  |  target: {TARGET}")
table.head(3)[["id", "formula", *HEADLINE_INPUTS, TARGET, "q", "alpha"]]

In [ ]:
assert len(table) == 133_885 and table["id"].is_unique
assert (table["mu"] >= 0).all(), "a dipole magnitude cannot be negative"
assert set(np.concatenate(table["Z"].to_numpy())) <= {1, 6, 7, 8, 9}

print(f"distinct formulas : {table['formula'].nunique()}")
print(f"atoms per molecule: {table['n_atoms'].min()}–{table['n_atoms'].max()}, "
      f"heavy atoms: {table['n_heavy'].min()}–{table['n_heavy'].max()}")
table.loc[table["id"].isin([1, 2, 3]), ["id", "formula", "n_atoms", "smiles"]]

## 2. Exclusions and duplicates

Rules apply in order, and the report counts how many IDs each rule *newly* removes, since the readme's hard-to-converge
list partly overlaps `uncharacterized.txt`.

The third rule is new in X2. QM9 lists a few molecules twice: same formula, same geometry (Coulomb-matrix spectrum
equal to 0.01) and the same |μ| (explore_01). Each such group keeps its smallest ID. Removing the extra copies before
splitting guarantees that no molecule can sit in two sets, or in a training fold and its validation fold. This uses
Z and R only. InChI cannot be used: standard InChI merges tautomers, which are different molecules with different
dipoles.

In [ ]:
from qm9dipole.cleaning import duplicate_extras, duplicate_groups
from qm9dipole.data import apply_exclusions, exclusion_rules
from qm9dipole.descriptors import cm_spectrum
from qm9dipole.splits import load_config

cfg = load_config()
rules = exclusion_rules(include_readme_flagged=cfg["exclude_readme_flagged"])
kept, _, _ = apply_exclusions(table, rules)
spectra = np.stack([cm_spectrum(z, r) for z, r in zip(kept["Z"], kept["R"])])
groups = duplicate_groups(kept["id"].to_numpy(), kept["formula"], spectra, cfg["duplicate_cm_decimals"])
extras = duplicate_extras(kept["id"].to_numpy(), kept["formula"], spectra, cfg["duplicate_cm_decimals"])
if cfg["exclude_geometric_duplicates"]:
    rules["geometric duplicate (extra copy)"] = extras
print(f"geometric duplicates: {len(groups)} molecules in {groups.nunique()} groups -> {len(extras)} extra copies")

kept, exclusion_report, excluded = apply_exclusions(table, rules)
print(f"kept {len(kept):,} of {len(table):,} molecules ({len(excluded):,} excluded), "
      f"{kept['formula'].nunique()} formulas")
exclusion_report

## 3. Build and check the splits

`build_splits` follows `src/qm9dipole/splits.py` step by step: whole-formula holdouts (test, then development) →
familiar test sample → development sample → training pool → quantum subsets and anchors → one fill order per
training seed. `check_splits` then runs every invariant, (a)–(i): disjoint sets, held-out formulas absent from
training, anchors in the smallest set, nesting, exact sizes, unique IDs, no excluded ID, complete holdouts with the
right formulas, and every kept molecule in exactly one set. Each check has a unit test that deliberately breaks it.

In [ ]:
from qm9dipole.provenance import config_hash
from qm9dipole.splits import build_splits, check_splits

splits = build_splits(kept, cfg)
check_splits(splits, table, excluded)
print(f"all split checks passed  (config_hash {config_hash(cfg)})")

## 4. Split report

Sizes, formulas and heavy-atom mix of every set. The unseen-formula holdout keeps X1's 93 formulas (its random draw
uses the same stream), but now holds all of their molecules.

In [ ]:
formula = table.set_index("id")["formula"]
n_heavy = table.set_index("id")["n_heavy"]

report = pd.DataFrame([
    {"set": name, "molecules": len(ids), "formulas": formula.loc[ids].nunique(),
     "share of kept": len(ids) / len(kept), "mean heavy atoms": n_heavy.loc[ids].mean()}
    for name, ids in splits.sets().items()
]).set_index("set")
display(report.style.format({"molecules": "{:,}", "share of kept": "{:.1%}", "mean heavy atoms": "{:.2f}"}))
print(f"unseen formulas: {len(splits.unseen_formulas)} (test), {len(splits.dev_unseen_formulas)} (development); "
      f"anchored familiar formulas: {len(splits.familiar_formulas)}")
heavy_mix = pd.DataFrame({name: n_heavy.loc[ids].value_counts(normalize=True)
                          for name, ids in splits.sets().items() if not name.endswith("_q")}).fillna(0).sort_index()
heavy_mix.index.name = "heavy atoms"
display(heavy_mix.T.style.format("{:.1%}").set_caption("heavy-atom mix of each set"))

### Training sets

One fill order of the training pool per seed: the anchors first, then a random permutation of the rest, so the
sets are nested by construction. The largest size is the whole pool (the same set for every seed). N = 100, 300
and 1000 are the quantum-comparable (Track B) sizes.

In [ ]:
fam = set(splits.familiar_formulas)
P_share = formula.loc[splits.pool].isin(fam).mean()
rows = []
for seed, by_n in splits.train.items():
    for n, ids in by_n.items():
        rows.append({"seed": seed, "N": n, "formulas": formula.loc[ids].nunique(),
                     "anchored formulas covered": len(fam & set(formula.loc[ids])),
                     "share from anchored formulas": formula.loc[ids].isin(fam).mean()})
train_report = pd.DataFrame(rows)
display(train_report.pivot(index="N", columns="seed", values="formulas").style.set_caption("formulas in S_(s,N)"))
display(train_report.pivot(index="N", columns="seed", values="share from anchored formulas").style.format("{:.0%}")
        .set_caption("share of S_(s,N) from the anchored familiar formulas"))
assert (train_report["anchored formulas covered"] == len(fam)).all()
print(f"anchored formulas' share of the training pool P: {P_share:.1%} (what a plain random sample would give)")

### Anchor skew (PLAN §4 note)

Anchors guarantee that every anchored familiar formula appears in the smallest training set, but they also
over-represent those formulas there. X1 anchored 40 formulas making up 9% of P, so 43–50% of S_(s,100) came from them.
X2 anchors 20 formulas chosen among those with several familiar test molecules (so common ones): the share above,
against the pool share, is the remaining skew. It fades by N = 1000.

## 5. The training pool

Heavy-atom counts and the target, on the training pool only (no test or development molecule). These plots are
descriptive; nothing is tuned on them.

In [ ]:
pool = table.set_index("id").loc[splits.pool]
by_heavy = pool.groupby("n_heavy").agg(molecules=("formula", "size"), formulas=("formula", "nunique"))
mu_cap = float(np.ceil(pool["mu"].quantile(0.995)))
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.8))
ax1.bar(by_heavy.index, by_heavy["molecules"], width=0.6, color=SERIES[0])
ax1.set(xticks=by_heavy.index, title="Training pool: molecules by heavy-atom count", xlabel="heavy atoms (C, N, O, F)",
        ylabel="molecules")
ax1.grid(axis="x", visible=False)
ax2.hist(pool["mu"].clip(upper=mu_cap), bins=np.arange(0, mu_cap + 0.25, 0.25), color=SERIES[0], edgecolor=SURFACE,
         linewidth=0.6)
ax2.set(title="Training pool: dipole magnitude |μ|", xlabel="|μ| (debye)", ylabel="molecules")
ax2.text(0.98, 0.95, f"last bar: {(pool['mu'] > mu_cap).sum()} molecules ≥ {mu_cap:.0f} D\n(max {pool['mu'].max():.1f} D)",
         transform=ax2.transAxes, ha="right", va="top", color=INK_SECONDARY, fontsize=9)
ax2.grid(axis="x", visible=False)
fig.tight_layout()
fig.savefig(REPO_ROOT / "figures" / "m1_pool_overview.png")
plt.show()
display(by_heavy.T)

## 6. Save the splits

One JSON file per set in `splits/`, each stamped with the split seed, `config_hash` and `git_hash`, plus one
`train_order_s{seed}.json` per training seed (S_(s,N) is its first N IDs). `meta.json` also records the config,
the formula lists, the exclusion counts and the excluded duplicate IDs. These files are committed, so every later
step uses exactly these molecules.

In [ ]:
from qm9dipole.splits import load_splits, save_splits

paths = save_splits(splits, extra_meta={
    "n_parsed": len(table),
    "n_kept": len(kept),
    "exclusions": json.loads(exclusion_report.to_json(orient="records")),
    "excluded_duplicates": sorted(int(i) for i in extras),
})
print(f"wrote {len(paths)} files to splits/:", ", ".join(p.name for p in paths))

reloaded = load_splits()
check_splits(reloaded, table, excluded)
assert all(np.array_equal(reloaded.all_id_arrays()[k], v) for k, v in splits.all_id_arrays().items())
print("reloaded splits are identical and pass all checks")

In [ ]:
import subprocess
import sys

r = subprocess.run([sys.executable, "-m", "pytest", "-q", "--color=no", "tests/test_parser.py", "tests/test_splits.py"],
                   cwd=REPO_ROOT, capture_output=True, text=True)
print(r.stdout[-1500:])
assert r.returncode == 0, "tests failed"

## M1 status (X2 design)

If every cell ran, M1's acceptance criteria are met for the redesigned splits. The test sets (`test_*`) are read
only by `scripts/final_eval.py`, once (CLAUDE.md rule 2). Model development uses the training sets and the two
development sets.

**Next:** `explore_01_eda` re-runs the exploratory analysis on the 99k-molecule training pool.